### Optimization

Machine learning, as we saw in the lecture, boils down to three ingredients:

1. a **model** with free **parameters** (here: a straight line),
2. a **cost (loss) function** that says how badly the model explains the data for a given set of parameters,
3. an **optimization** procedure that searches for the parameters with the lowest cost.

In this notebook we go through all three on the simplest possible example: fitting a line to noisy points. Since the line has only two parameters, we can afford to evaluate the cost for *every* combination of them and look at the whole **cost surface** at once. In lesson 7 the model was an affine transform (6 parameters) or a homography (8 parameters); the principle is exactly the same, we just cannot draw its cost surface.

In [ ]:
import cv2
import math
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

### The data

Our model is a line $y = ax + b$. We describe it by two parameters: the angle $\alpha$ of the line (so that $a = \tan\alpha$) and the offset $b$, i.e. the value of $y$ at $x = 0$. We generate the points around a known **target** line (45°, offset 1), so that we know what the optimization should find. There are 500 noisy inliers that follow the line and, on top of them, 50 outliers with a 25 times larger noise, which do not follow the model at all. Real data always contains some of those.

In [ ]:
# Target parameters of the line we want to recover
target_angle = 45
target_slope = np.tan(np.deg2rad(target_angle))
target_offset = 1

np.random.seed(0)  # Fixed seed: the same data (and the same plots) in every run

# Let's contaminate target data with noise
x_data = np.arange(0, 100, 0.2)
y_data = target_slope*x_data + target_offset + np.random.randn(len(x_data))

# Add unrelated samples (outliers) to dataset
x_noise = np.arange(0, 100, 2)
y_noise = target_slope*x_noise + target_offset + 25*np.random.randn(len(x_noise))

plt.plot(x_data, y_data, 'o', label='inliers')
plt.plot(x_noise, y_noise, 'o', label='outliers')
plt.xlabel('x'), plt.ylabel('y')
plt.legend(), plt.grid(True)

# Mix true (noisy) data and outliers
x = np.concatenate((x_data, x_noise))
y = np.concatenate((y_data, y_noise))

print('Amount of inliers: ', len(x_data))
print('Amount of outliers:', len(x_noise))

### The cost function

For a candidate line we need a single number that tells us how well it fits the points. We use the **mean distance** between the points and the line, measured perpendicular to the line. For a line $Ax + By + C = 0$ the distance to a point $(x_0, y_0)$ is

$$ d = \frac{|A x_0 + B y_0 + C|}{\sqrt{A^2 + B^2}} $$

Our line $y = \tan(\alpha)\,x + b$ can be written as $-\tan(\alpha)\,x + y - b = 0$, i.e. $A = -\tan\alpha$, $B = 1$ and $C = -b$.

The cost depends on just two variables, $\alpha$ and $b$, so we can compute it by brute force on a grid: all angles from 0° to 359° in steps of 1° and all offsets from -100 to 99 in steps of 1. That makes 360 × 200 = 72 000 candidate lines, each one evaluated on all 550 points.

In [ ]:
# Define distance from point to line
def distance(a, b, c, x_point, y_point):
    numerator = np.abs(a*x_point + b*y_point + c)
    denominator = np.sqrt(a**2 + b**2)
    return numerator/denominator

alphas = np.arange(0, 360)      # Candidate angles [deg]
offsets = np.arange(-100, 100)  # Candidate offsets
cost = np.zeros((len(offsets), len(alphas)))

for idx_alpha in range(len(alphas)):
    for idx_offset in range(len(offsets)):
        slope = np.tan(np.deg2rad(alphas[idx_alpha]))
        cost[idx_offset, idx_alpha] = np.mean(distance(-slope, 1, -offsets[idx_offset], x, y))

### Reading the cost surface

Every pixel of the image below is one candidate line: the horizontal axis is its angle, the vertical axis its offset and the brightness is its cost (**dark = low cost**). We show the *logarithm* of the cost. The costs span a wide range (from about 2 to about 158) and the logarithm keeps the differences between the good candidates visible.

In [ ]:
plt.imshow(np.log(cost), cmap='gray', aspect='auto', extent=[-0.5, 359.5, 99.5, -100.5])
plt.xlabel('Angle [deg]'), plt.ylabel('Offset')

What can we read from the picture?

* **The low-cost candidates form a dark valley.** There is one best combination of angle and offset and the cost grows as we move away from it. Optimization means finding the bottom of this valley.
* **There are two identical valleys**, one around 45° and one around 225°. It is not a coincidence: a line with angle $\alpha$ and a line with angle $\alpha + 180°$ are the very same line, so the cost surface repeats itself every 180°.
* **The valley is curved instead of vertical.** The offset $b$ is measured at $x = 0$, far away from where the data live (around $x \approx 50$). When we tilt the line, the offset must change a lot to keep the line among the points: the best offset for a given angle follows $b \approx \bar{y} - \tan(\alpha)\,\bar{x}$, i.e. the line passes through the center of the points. Narrow, curved valleys like this are harder for optimization algorithms that follow the slope of the cost.

### Finding the minimum

The brute-force optimization is now trivial: we just pick the candidate with the lowest cost. Since the second valley is a copy of the first one, it is enough to search the angles below 180°.

In [ ]:
idx_offset, idx_alpha = np.unravel_index(np.argmin(cost[:, :180]), cost[:, :180].shape)
best_alpha, best_offset = alphas[idx_alpha], offsets[idx_offset]

print('Best angle: ', best_alpha)
print('Best offset:', best_offset)
print('Lowest cost:', cost[idx_offset, idx_alpha])

The optimization recovered the target line (45°, offset 1) exactly, up to the resolution of our grid, in spite of the noise and the outliers. Let's mark the minima (both valleys) on the cost surface and draw the best line over the data.

In [ ]:
plt.subplot(211)
plt.imshow(np.log(cost), cmap='gray', aspect='auto', extent=[-0.5, 359.5, 99.5, -100.5])
plt.plot([best_alpha, best_alpha + 180], [best_offset, best_offset], 'r+', markersize=15)
plt.xlabel('Angle [deg]'), plt.ylabel('Offset')

plt.subplot(212)
plt.plot(x_data, y_data, 'o'), plt.plot(x_noise, y_noise, 'o')
x_line = np.array([0, 100])
plt.plot(x_line, np.tan(np.deg2rad(best_alpha))*x_line + best_offset, 'r', linewidth=3)
plt.xlabel('x'), plt.ylabel('y'), plt.grid(True)

### Brute force does not scale

We found the minimum by trying every candidate: 72 000 cost evaluations for just two parameters. Every additional parameter multiplies the number of candidates by the number of values tried for it. Even with only 100 values per parameter, the 8 parameters of a homography would need $100^8 = 10^{16}$ evaluations, and a deep neural network has millions of parameters.

Real optimizers therefore do not try everything. They start somewhere on the cost surface, look at which direction leads downhill and take a step that way. This idea is called **gradient descent** and we will get to it in lesson 12. K-means, the topic of the next notebook, is another iterative optimizer of this kind.